# HeH⁺ potential energy surface: from exact references to a noisy quantum device
**Team:** *Name A* (algorithm & physics), *Name B* (hardware & noise)

**BasQ Qiskit Fall Fest 2026, Challenge 1, Beginner track**

## Abstract
We compute the potential energy surface of the HeH⁺ molecular ion with VQE and follow it all the way from exact classical references to a noisy quantum device. Exploiting electron-number symmetries (parity mapping + ℤ₂ tapering) shrinks the STO-3G problem from 4 to 2 qubits with no loss of accuracy, which cuts the compiled circuit on an IBM Heron device from 45 to 2 two-qubit gates. Noiseless VQE matches exact diagonalisation to < 4×10⁻⁷ Ha at all 14 distances (itself cross-checked against PySCF FCI to 10⁻¹⁴ Ha), giving r_eq = 0.914 Å and D_e = 1.49 eV in STO-3G. A 6-31G VQE (6 qubits) and a classical FCI basis-set ladder up to cc-pVQZ (0.775 Å, 2.04 eV, matching the literature) show that the basis set, not the quantum algorithm, dominates the error. Under the gate noise of a real device (FakeFez), raw errors are 5–10 mHa; a noise-aware qubit layout plus zero-noise extrapolation brings them to 1.4–2.6 mHa, reaching chemical accuracy at 1.5 and 2.5 Å. Readout error adds about 3 mHa that ZNE cannot remove. A complete real-hardware pipeline (one job, dynamical decoupling, twirling, TREX and ZNE) is validated end-to-end in a 32768-shot dry run on the fake device; we did not run it on a real QPU within the time available, and the dry run shows why chemical accuracy could not be claimed there anyway: shot noise alone is ≈ 4.5 mHa per energy. The main limitations are shot noise on hardware (an optimiser loop would need SPSA and hundreds of evaluations; gradient-based SLSQP fails outright under noise) and the steep growth of qubits and gates with basis size.

## Reproducibility
```bash
pip install -r requirements.txt          # pinned versions
python beginner/check_setup.py           # must print OK
```
Run top to bottom (≈ 3–4 min on a laptop without hardware). Nothing is sent to IBM hardware unless `RUN_HARDWARE = True` (Section 7). All energies are **total energies in Hartree**. Chemical accuracy = 1.6 mHa.

# 0. Setup

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from pes_tools import (CHEMICAL_ACCURACY, HARTREE_TO_KCAL_MOL, build_problem,
                       hf_energy, exact_energy, vqe_energy, uccsd_ansatz)
from hardware_tools import (make_mapper, prepare_point, transpile_report, best_transpile,
                            noisy_estimator, estimate, fold_two_qubit_gates, zne_extrapolate,
                            runtime_estimator)

warnings.filterwarnings("ignore", module="scipy")
RESULTS, FIGURES = Path("results"), Path("figures")
RUN_HARDWARE = False  # flip to True only to (re)submit real-device jobs

# Distance grid (Angstrom): dense around the minimum, coarser towards dissociation
DISTANCES = np.round(np.concatenate([np.linspace(0.5, 1.2, 8), np.linspace(1.4, 3.5, 6)]), 3)
HW_DISTANCES = [0.6, 0.774, 1.0, 1.5, 2.5]  # subset for noisy sim / hardware (0.774 Å = literature r_eq)

# 1. System and qubit encoding

HeH⁺: 2 electrons (He 2 + H 1 − 1 for the charge), closed-shell singlet. In STO-3G that is 2 spatial orbitals, i.e. 4 spin-orbitals.

**Claim:** exploiting the electron-number symmetries shrinks the problem from 4 qubits (Jordan-Wigner) to 2 (parity mapping + two-qubit reduction) with **no loss of accuracy**, and that is what makes the hardware runs in Sections 5–7 feasible.

**Why the reduction is exact.** The molecular Hamiltonian conserves the number of spin-up electrons N↑ and spin-down electrons N↓. In the parity encoding, qubit *k* stores the parity of the occupations of spin-orbitals 0…*k*, so the qubit at the end of the spin-up block stores (−1)^N↑ and the last qubit stores (−1)^(N↑+N↓). Both are fixed numbers for our molecule (N↑ = N↓ = 1), so those two qubits are never flipped by the Hamiltonian: every Pauli term acts on them with only I or Z. We replace those Z's by their known eigenvalues (±1) and delete the two qubits. The 2-qubit Hamiltonian has exactly the same spectrum as the 4-qubit one *inside* the correct symmetry sector, which is the sector we want. This is the ℤ₂ "qubit tapering" of Bravyi, Gambetta, Mezzacapo & Temme, *arXiv:1701.08213* (2017).

**Why HeH⁺ stops at 2 qubits while H₂ goes down to 1.** Qiskit's tapering also searches for any *extra* ℤ₂ symmetries of the qubit Hamiltonian. H₂ has one, inversion symmetry (the two atoms are identical), so it tapers to a single qubit. HeH⁺ is heteronuclear, so there is no inversion centre and no extra symmetry: `tapered` gives the same 2 qubits as `parity` (checked in the cell below).

In [ ]:
problem = build_problem(0.774)
rows = []
for kind in ["jw", "parity", "tapered"]:
    m = make_mapper(problem, kind)
    op = m.map(problem.hamiltonian.second_q_op())
    a = uccsd_ansatz(problem, m)
    rows.append({"encoding": kind, "qubits": op.num_qubits, "pauli_terms": len(op),
                 "uccsd_params": a.num_parameters, "exact_Ha": exact_energy(problem, m)})
encodings = pd.DataFrame(rows); encodings

In [ ]:
# Same pipeline on H2 (identical atoms -> extra inversion symmetry) for contrast
from qiskit_nature.second_q.drivers import PySCFDriver
h2 = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto3g").run()
print("H2   qubits:", {k: make_mapper(h2, k).map(h2.hamiltonian.second_q_op()).num_qubits for k in ["jw", "parity", "tapered"]})
print("HeH+ qubits:", dict(zip(encodings.encoding, encodings.qubits)))
print("Max spread of exact energies across encodings:", f"{np.ptp(encodings.exact_Ha):.1e} Ha")
print("\nThe whole 2-qubit HeH+ Hamiltonian at 0.774 Å (electronic part, Ha):")
print(make_mapper(problem, "tapered").map(problem.hamiltonian.second_q_op()))

**Reading the table.** Jordan-Wigner needs 4 qubits and 27 Pauli terms; the 2-qubit encoding needs only 9 terms, and the exact energy is the same in all three encodings (identical to machine precision). That confirms the reduction is lossless. Fewer qubits mean shallower routed circuits (Section 5), and fewer Pauli terms mean fewer measurement settings on hardware. UCCSD always has 3 parameters here: two single excitations (one per spin) and one double excitation that moves both electrons from the bonding to the antibonding orbital. `parity` and `tapered` coincide because HeH⁺ has no spatial symmetry left to remove, unlike H₂ (1 qubit).

# 2. Classical references and an independent check

Before trusting any quantum number, we check our "exact" reference two independent ways: Qiskit Nature's diagonalisation and PySCF's own FCI solver.

**Pitfall we found:** the tutorial's unfiltered `NumPyMinimumEigensolver` searches every electron count. For HeH⁺ it returns a state with extra electrons, ≈0.16 Ha too low. We filter on particle number and spin (`exact_energy` in `pes_tools.py`).

In [ ]:
from pyscf import gto, scf, fci
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

def pyscf_fci(d, basis="sto3g"):
    mol = gto.M(atom=f"He 0 0 0; H 0 0 {d}", basis=basis, charge=1, spin=0, verbose=0)
    mf = scf.RHF(mol).run()
    return mf.e_tot, fci.FCI(mf).kernel()[0]

e_rhf, e_fci = pyscf_fci(0.774)
p = build_problem(0.774)
unf = GroundStateEigensolver(make_mapper(p, "jw"), NumPyMinimumEigensolver()).solve(p)
unfiltered = unf.total_energies[0]
print(f"PySCF RHF          {e_rhf: .8f}   Qiskit HF circuit {hf_energy(p): .8f}")
print(f"PySCF FCI          {e_fci: .8f}   Qiskit exact      {exact_energy(p): .8f}")
print(f"Unfiltered (wrong) {unfiltered: .8f}   off by {unfiltered - e_fci:+.3f} Ha, "
      f"electrons in that state = {unf.num_particles[0]:.0f} (should be 2)")

# Same cross-check over the whole distance grid, not just one point
check = pd.DataFrame([{"distance_A": d, "pyscf_fci": pyscf_fci(d)[1],
                       "qiskit_exact": exact_energy(build_problem(float(d)), make_mapper(build_problem(float(d)), "tapered"))}
                      for d in DISTANCES])
check["diff"] = (check.pyscf_fci - check.qiskit_exact).abs()
check.to_csv(RESULTS / "fci_crosscheck.csv", index=False)
print(f"\nMax |PySCF FCI - Qiskit tapered exact| over {len(check)} distances: {check['diff'].max():.1e} Ha")
assert check["diff"].max() < 1e-8

**What this shows.** Two independent codes (PySCF's FCI solver and Qiskit's diagonalisation of the *tapered 2-qubit* Hamiltonian) agree to ~10⁻¹⁴ Ha at every distance. That validates the reference, the integrals, the fermion-to-qubit mapping and the tapering in one go. Qiskit's HF circuit also reproduces PySCF's RHF energy exactly, so the starting point of VQE is right.

The unfiltered solver returns a state with **3 electrons** (neutral HeH, about 0.16 Ha too low). The qubit Hamiltonian lives in Fock space, where all electron numbers coexist, and more electrons can mean lower energy. It went unnoticed in the H₂ tutorial only because there the 2-electron state happened to be the lowest. Every "exact" number in this notebook is filtered (or tapered, which fixes the electron number by construction).

# 3. Noiseless potential energy surface

Noiseless VQE (UCCSD, tapered 2 qubits, SLSQP) against HF and exact over the whole curve. Each point is warm-started from the previous one's parameters, and Section 4c compares that with cold starts.

In [ ]:
rows, prev = [], None
for d in DISTANCES:
    prob = build_problem(float(d)); m = make_mapper(prob, "tapered")
    v = vqe_energy(prob, mapper=m, initial_point=prev)
    prev = v.optimal_point
    rows.append({"distance_A": d, "hf": hf_energy(prob, m), "vqe": v.energy,
                 "exact": exact_energy(prob, m), "evals": v.cost_function_evals})
pes = pd.DataFrame(rows)
pes["vqe_err"] = (pes.vqe - pes.exact).abs()
pes["correlation"] = pes.exact - pes.hf
pes.to_csv(RESULTS / "pes_noiseless_sto3g.csv", index=False)
pes

In [ ]:
fig, (ax, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 9), sharex=True, height_ratios=[3, 1.2, 1.2])
ax.plot(pes.distance_A, pes.hf, "o--", c="#9e9e9e", label="Hartree-Fock")
ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(pes.distance_A, pes.vqe, "x", c="#1f77b4", ms=9, label="VQE (UCCSD, 2 qubits)")
ax.set_ylabel("Total energy (Ha)"); ax.legend(); ax.set_title("HeH$^+$ PES, STO-3G, noiseless")
ax2.semilogy(pes.distance_A, pes.vqe_err, "o-", c="#1f77b4", label="|VQE − exact|")
ax2.axhline(CHEMICAL_ACCURACY, c="red", ls=":", label="chemical accuracy")
ax2.set_ylim(1e-12, 1e-2); ax2.set_ylabel("Error (Ha)"); ax2.legend(loc="lower right", fontsize=8)
ax3.plot(pes.distance_A, 1e3 * pes.correlation, "s-", c="#d62728")
ax3.axhline(-1e3 * CHEMICAL_ACCURACY, c="red", ls=":")
ax3.set_ylabel("E$_{corr}$ (mHa)"); ax3.set_xlabel("He–H distance (Å)")
fig.tight_layout(); fig.savefig(FIGURES / "pes_noiseless_sto3g.png", dpi=150)

### 3a. Equilibrium bond length

In [ ]:
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar
spl = CubicSpline(pes.distance_A, pes.vqe)
opt = minimize_scalar(spl, bounds=(pes.distance_A.min(), pes.distance_A.max()), method="bounded")
r_spline = opt.x
print(f"r_eq (VQE, spline over grid) = {r_spline:.4f} Å")

# Refine: minimise the VQE energy itself (one VQE run per trial distance), warm-started
p0 = build_problem(r_spline)
warm = vqe_energy(p0, mapper=make_mapper(p0, "tapered")).optimal_point   # warm start for the refinement
def vqe_at(d):
    prob = build_problem(float(d))
    return vqe_energy(prob, mapper=make_mapper(prob, "tapered"), initial_point=warm).energy
ref = minimize_scalar(vqe_at, bounds=(r_spline - 0.05, r_spline + 0.05), method="bounded", options={"xatol": 1e-4})
r_eq, e_eq = ref.x, float(ref.fun)
p_eq = build_problem(r_eq)
e_eq_exact = exact_energy(p_eq, make_mapper(p_eq, "tapered"))
R_EQ_LIT, DE_LIT_EV = 0.7743, 2.040   # Kolos & Peek, Chem. Phys. 12, 381 (1976): R_e = 1.4632 bohr, D_e = 2.040 eV
print(f"r_eq (VQE, direct)           = {r_eq:.4f} Å,  E_VQE = {e_eq:.6f} Ha,  |E_VQE - E_exact| = {abs(e_eq - e_eq_exact):.1e} Ha")
print(f"Literature (near-exact)      = {R_EQ_LIT:.4f} Å   ->  STO-3G error = {r_eq - R_EQ_LIT:+.3f} Å ({100 * (r_eq / R_EQ_LIT - 1):+.0f} %)")

### 3b. Dissociation

In [ ]:
def atom_fci(symbol, charge, spin, basis="sto3g"):
    nelec = {"He": 2, "H": 1}[symbol] - charge
    if nelec == 0:
        return 0.0  # a bare proton has no electrons
    mol = gto.M(atom=f"{symbol} 0 0 0", basis=basis, charge=charge, spin=spin, verbose=0)
    mf = (scf.RHF if spin == 0 else scf.ROHF)(mol).run()
    return fci.FCI(mf).kernel()[0]

channels = {"He + H+": atom_fci("He", 0, 0) + atom_fci("H", 1, 0),
            "He+ + H": atom_fci("He", 1, 1) + atom_fci("H", 0, 1)}
for k, v in channels.items(): print(f"{k:8s} {v: .6f} Ha")
print(f"Gap between channels: {channels['He+ + H'] - channels['He + H+']:.3f} Ha "
      f"(real gap = IE(He) - IE(H) = 24.59 - 13.60 eV = {(24.587 - 13.598) / 27.2114:.3f} Ha)")
e_inf = min(channels.values())
print(f"Long-distance end of our curve: {pes.exact.iloc[-1]:.6f} Ha  "
      f"(only {1e6 * (e_inf - pes.exact.iloc[-1]):.1f} µHa below He + H+, so converged at {pes.distance_A.iloc[-1]} Å)")
D_e = e_inf - e_eq
print(f"D_e (STO-3G, VQE) = {D_e:.4f} Ha = {D_e * HARTREE_TO_KCAL_MOL:.1f} kcal/mol = {D_e * 27.2114:.2f} eV   "
      f"vs literature {DE_LIT_EV} eV")

**Interpretation of Section 3.**
- **VQE is essentially exact on the noiseless simulator.** The worst |VQE − exact| over the 14 distances is ~3×10⁻⁷ Ha, about 5000× below chemical accuracy. The tapered space has only 4 states and UCCSD's 3 rotations can reach the exact ground state in it, so the only error left is the optimiser's stopping tolerance. Warm-starting keeps every point to ≤ 18 energy evaluations.
- **Correlation energy is small and lives at short distance.** E_exact − E_HF peaks at about −10 mHa around 0.7 Å, which is six times chemical accuracy, so Hartree-Fock alone would *not* be good enough near equilibrium. It falls below 1.6 mHa beyond ~1.5 Å and vanishes at dissociation.
- **Why it vanishes (and why this differs from H₂).** HeH⁺ dissociates into He + H⁺: a closed-shell atom plus a bare proton. In STO-3G He has a single 1s orbital, so its two electrons have nowhere else to go and HF is exact there. For H₂, restricted HF fails badly at dissociation because it forces ionic H⁻ + H⁺ character. HeH⁺ is a "well-behaved" case for HF, and correlation matters mostly in the bonding region.
- **Dissociation channel.** The electrons stay on helium: He + H⁺ is lower than He⁺ + H because helium holds its electrons much more tightly (ionisation energy 24.6 eV vs 13.6 eV). STO-3G gets the order right. The curve is flat to a few µHa by 3.5 Å, so our grid reaches the asymptote *in this basis*. In reality, H⁺ polarises He and gives an attractive −α/(2R⁴) tail (≈ −0.4 mHa at 3.5 Å), which a basis with no p-functions cannot describe.
- **The real limitation is the basis, not the quantum algorithm.** STO-3G puts the minimum at ≈ 0.91 Å (+18 % vs the true 0.774 Å) and underestimates D_e (≈ 1.5 eV vs 2.04 eV). Section 4 shows these errors are ~10⁵× larger than the VQE error and that a bigger basis removes them.

# 4. Beyond the baseline

Three studies, each answering one question:
- **4a. Basis set.** How much of our error comes from STO-3G itself? We run VQE in 6-31G (4 spatial orbitals → 6 tapered qubits, 15 UCCSD parameters) over a coarse grid, then compare with a purely classical FCI "basis-set ladder" up to cc-pVQZ to reach the near-exact answer.
- **4b. Ansatz.** UCCSD (chemistry-inspired) vs hardware-efficient circuits (`real_amplitudes`, `efficient_su2`): parameters, CNOT count, and robustness to random starting points.
- **4c. Optimiser.** SLSQP vs COBYLA vs SPSA, noiseless and with simulated shot noise, plus warm vs cold starts along the curve.

In [ ]:
rows = []
for basis in ["sto3g", "6-31g"]:
    for kind in ["jw", "tapered"]:
        t0 = time.perf_counter()
        prob = build_problem(0.774, basis=basis); m = make_mapper(prob, kind)
        v = vqe_energy(prob, mapper=m)
        rows.append({"basis": basis, "encoding": kind, "qubits": v.num_qubits, "params": v.num_parameters,
                     "vqe": v.energy, "exact": exact_energy(prob, m), "seconds": time.perf_counter() - t0})
scaling = pd.DataFrame(rows); scaling["err"] = (scaling.vqe - scaling.exact).abs()
scaling.to_csv(RESULTS / "basis_encoding_scaling.csv", index=False); scaling

In [ ]:
# 4a. 6-31G potential energy surface with VQE (6 tapered qubits, 15 parameters)
GRID_631G = [0.6, 0.7, 0.75, 0.8, 0.9, 1.1, 1.5, 2.5, 3.5]
rows, prev = [], None
for d in GRID_631G:
    t0 = time.perf_counter()
    prob = build_problem(d, basis="6-31g"); m = make_mapper(prob, "tapered")
    v = vqe_energy(prob, mapper=m, initial_point=prev); prev = v.optimal_point
    rows.append({"distance_A": d, "hf": hf_energy(prob, m), "vqe": v.energy, "exact": exact_energy(prob, m),
                 "evals": v.cost_function_evals, "seconds": time.perf_counter() - t0})
pes631 = pd.DataFrame(rows)
pes631["vqe_err"] = (pes631.vqe - pes631.exact).abs(); pes631["correlation"] = pes631.exact - pes631.hf
pes631.to_csv(RESULTS / "pes_noiseless_631g.csv", index=False)
r631 = minimize_scalar(CubicSpline(pes631.distance_A, pes631.vqe), bounds=(0.6, 1.1), method="bounded")
De631 = atom_fci("He", 0, 0, basis="6-31g") - r631.fun
print(f"6-31G VQE: r_eq = {r631.x:.4f} Å, D_e = {De631 * 27.2114:.2f} eV, max |VQE-exact| = {pes631.vqe_err.max():.1e} Ha, "
      f"~{pes631.seconds.mean():.1f} s/point")
pes631

In [ ]:
# 4a (cont.). Classical FCI basis-set ladder: where does the true answer lie?
def fci_total(d, basis):
    mol = gto.M(atom=f"He 0 0 0; H 0 0 {d}", basis=basis, charge=1, spin=0, verbose=0)
    return fci.FCI(scf.RHF(mol).run()).kernel()[0], mol.nao

rows = []
for basis in ["sto3g", "6-31g", "6-31g**", "cc-pvdz", "cc-pvtz", "cc-pvqz"]:
    o = minimize_scalar(lambda d: fci_total(d, basis)[0], bounds=(0.6, 1.1), method="bounded", options={"xatol": 1e-4})
    nao = fci_total(0.8, basis)[1]
    De = atom_fci("He", 0, 0, basis=basis) - o.fun
    rows.append({"basis": basis, "spatial_orbitals": nao, "qubits_JW": 2 * nao, "qubits_tapered": 2 * nao - 2,
                 "r_eq_A": o.x, "E_eq_Ha": o.fun, "D_e_eV": De * 27.2114})
ladder = pd.DataFrame(rows)
ladder["r_eq_err_A"] = ladder.r_eq_A - R_EQ_LIT; ladder["D_e_err_eV"] = ladder.D_e_eV - DE_LIT_EV
ladder.to_csv(RESULTS / "basis_ladder_fci.csv", index=False)
print(f"Literature: r_eq = {R_EQ_LIT} Å, D_e = {DE_LIT_EV} eV (Kolos & Peek 1976)")
ladder.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for df, lab, c in [(pes, "STO-3G (2 qubits)", "#1f77b4"), (pes631, "6-31G (6 qubits)", "#ff7f0e")]:
    ax.plot(df.distance_A, df.exact, "-", c=c, alpha=0.6)
    ax.plot(df.distance_A, df.vqe, "o", c=c, label=f"VQE {lab}")
    ax.plot(df.distance_A, df.hf, "--", c=c, alpha=0.4)
qz = ladder.set_index("basis").loc["cc-pvqz"]
ax.plot(qz.r_eq_A, qz.E_eq_Ha, "k*", ms=14, label="FCI cc-pVQZ minimum (classical)")
ax.axvline(R_EQ_LIT, c="k", ls=":", lw=1, label="literature r$_{eq}$")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)")
ax.set_title("Basis set dominates the error: VQE (dots) sits on exact (lines), HF dashed"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "pes_basis_comparison.png", dpi=150)

### 4b. Ansatz comparison (STO-3G, tapered, at our r_eq)

In [ ]:
from qiskit import transpile
from qiskit.circuit.library import efficient_su2, real_amplitudes
from qiskit_nature.second_q.circuit.library import HartreeFock

prob = build_problem(r_eq); m = make_mapper(prob, "tapered"); e_ref = exact_energy(prob, m)
hf_circ = HartreeFock(prob.num_spatial_orbitals, prob.num_particles, m)
ansatze = {"UCCSD": uccsd_ansatz(prob, m),
           "real_amplitudes reps=1": hf_circ.compose(real_amplitudes(2, reps=1)),
           "efficient_su2 reps=1": hf_circ.compose(efficient_su2(2, reps=1)),
           "efficient_su2 reps=2": hf_circ.compose(efficient_su2(2, reps=2))}
rng = np.random.default_rng(7)
N_RANDOM = 4
rows = []
for name, a in ansatze.items():
    flat = transpile(a.decompose(reps=3), basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    errs, evals = [], []
    for trial in range(1 + N_RANDOM):   # trial 0 starts at zero parameters, the rest at random points
        x0 = np.zeros(a.num_parameters) if trial == 0 else rng.uniform(-np.pi, np.pi, a.num_parameters)
        v = vqe_energy(prob, mapper=m, ansatz=a, initial_point=x0)
        errs.append(abs(v.energy - e_ref)); evals.append(v.cost_function_evals)
    rows.append({"ansatz": name, "params": a.num_parameters, "cnots": flat.count_ops().get("cx", 0), "depth": flat.depth(),
                 "err_zero_start": errs[0], "worst_err_random_start": max(errs[1:]), "mean_evals": np.mean(evals)})
ansatz_cmp = pd.DataFrame(rows); ansatz_cmp.to_csv(RESULTS / "ansatz_comparison.csv", index=False); ansatz_cmp

### 4c. Optimiser comparison under shot noise, and warm vs cold starts

On hardware every energy is an *estimate* from finite shots. We mimic that with `StatevectorEstimator(default_precision=σ)`, which adds Gaussian noise of standard deviation σ (Ha) to every energy evaluation. σ = 0.001 Ha needs roughly 10⁵–10⁶ shots per evaluation for this Hamiltonian. We score each optimiser by the **true** (noiseless) energy at the parameters it returns, not by the noisy value it reports.

*Implementation detail that bit us:* passing an integer `seed` makes the estimator restart its random generator on every call, so the "noise" becomes the same constant offset each time and cannot disturb the optimiser. We pass a persistent `np.random.Generator` instead.

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import SLSQP, COBYLA, SPSA
from qiskit_algorithms.utils import algorithm_globals

a = uccsd_ansatz(prob, m); op = m.map(prob.hamiltonian.second_q_op())
true_energy = lambda x: float(StatevectorEstimator().run([(a, op, x)]).result()[0].data.evs) + prob.nuclear_repulsion_energy
optimizers = {"SLSQP": lambda: SLSQP(), "COBYLA": lambda: COBYLA(maxiter=200), "SPSA": lambda: SPSA(maxiter=300)}
rows = []
for sigma in [0.0, 0.001, 0.005]:
    for name, make in optimizers.items():
        for seed in range(3 if sigma else 1):
            algorithm_globals.random_seed = seed          # SPSA's own random perturbations
            est = StatevectorEstimator(default_precision=sigma, seed=np.random.default_rng(seed))
            v = vqe_energy(prob, mapper=m, ansatz=a, optimizer=make(), estimator=est)
            rows.append({"sigma_Ha": sigma, "optimizer": name, "seed": seed, "evals": v.cost_function_evals,
                         "true_err": abs(true_energy(v.optimal_point) - e_ref), "reported_minus_exact": v.energy - e_ref})
opt_cmp = pd.DataFrame(rows); opt_cmp.to_csv(RESULTS / "optimizer_comparison.csv", index=False)
summary = opt_cmp.groupby(["sigma_Ha", "optimizer"]).agg(evals=("evals", "mean"), median_true_err=("true_err", "median"),
                                                         worst_true_err=("true_err", "max"),
                                                         mean_reported_minus_exact=("reported_minus_exact", "mean"))
summary["within_chem_acc"] = summary.worst_true_err < CHEMICAL_ACCURACY
summary

In [ ]:
# Warm vs cold starts along the whole STO-3G curve (Section 3 used warm starts)
cold = []
for d in DISTANCES:
    prob_d = build_problem(float(d)); v = vqe_energy(prob_d, mapper=make_mapper(prob_d, "tapered"))
    cold.append({"distance_A": d, "cold_evals": v.cost_function_evals, "cold_vqe": v.energy})
warm_cold = pes[["distance_A", "evals", "vqe"]].rename(columns={"evals": "warm_evals", "vqe": "warm_vqe"}).merge(pd.DataFrame(cold))
warm_cold.to_csv(RESULTS / "warm_vs_cold.csv", index=False)
print(f"Total evaluations over {len(DISTANCES)} points: warm {warm_cold.warm_evals.sum()}, cold {warm_cold.cold_evals.sum()} "
      f"({100 * (1 - warm_cold.warm_evals.sum() / warm_cold.cold_evals.sum()):.0f} % saved); "
      f"max energy difference {np.abs(warm_cold.warm_vqe - warm_cold.cold_vqe).max():.1e} Ha")

**Interpretation of Section 4.**
- **Basis-set error dwarfs everything else.** At equilibrium, going from STO-3G to 6-31G lowers the energy by ~80 mHa and moves r_eq from ≈ 0.91 Å to ≈ 0.77 Å, while VQE stays within ~10⁻⁷ Ha of exact in *both* bases. So for a noiseless simulation the quantum algorithm is not the bottleneck; the basis is. The classical ladder converges to r_eq ≈ 0.775 Å and D_e ≈ 2.04 eV in cc-pVQZ, matching the literature (0.774 Å, 2.04 eV) and independently validating those reference values.
- **The STO-3G picture of correlation was partly an artefact.** In 6-31G the correlation energy is about −22 mHa near equilibrium (twice STO-3G) and does *not* vanish at dissociation (−15 mHa at 3.5 Å): helium now has a second s orbital, so its own two electrons can correlate. In a minimal basis an isolated He atom has zero correlation energy; the real value is about −42 mHa.
- **Polarisation functions matter for binding.** 6-31G already fixes r_eq but still underestimates D_e (≈ 1.7 eV). Adding p-functions (6-31G**) brings D_e to ≈ 2.0 eV: the bond in HeH⁺ is largely the proton polarising helium's electron cloud, which s-only bases cannot describe.
- **The price is qubits and run time.** 6-31G needs 6 tapered qubits (8 in JW) and 15 UCCSD parameters; each VQE point costs ~70–120 evaluations and 5–9 s instead of ~15 evaluations and 0.2 s. Tapering also pays off classically: the 8-qubit JW run takes ~30 s vs ~9 s tapered. cc-pVQZ would need 118 tapered qubits, far beyond what VQE on today's hardware can handle, while classical FCI does it in seconds for 2 electrons. *HeH⁺ is a validation problem, not one where a quantum computer has an advantage.*
- **Ansatz:** all four ansätze reach the exact energy from a zero start. The hardware-efficient `real_amplitudes` circuit needs only **1 CNOT** (UCCSD: 4 after generic compilation, 2 after the device transpiler's level-2/3 optimisation in Section 5), because any real 2-qubit state can be made with one CNOT, and after tapering every 2-qubit state already has the right electron number, so a hardware-efficient ansatz cannot "leak" into unphysical states here. The cost is optimisation effort: `efficient_su2` has 8–12 parameters, needs 5–7× more evaluations than UCCSD, and from random starts the reps=1 version stopped ~10⁻⁴ Ha short of the minimum (still within chemical accuracy here). That is the classic trainability trade-off of hardware-efficient circuits, and it worsens quickly with size.
- **Optimiser under shot noise:** noiselessly, SLSQP is the fastest (~14 evaluations). With shot noise it **fails badly** (errors of ~1 Ha): its finite-difference gradients use tiny steps, so they are pure noise. Gradient-free COBYLA uses few evaluations but stops once noise hides further progress, ending a few mHa off (median ≈ 2 mHa at σ = 1 mHa, worst ≈ 9 mHa at σ = 5 mHa), so it misses chemical accuracy. SPSA, designed for noisy objectives, was the only optimiser within chemical accuracy in every noisy run (worst ≈ 0.9 mHa), at the price of 600 evaluations.
- **Winner's curse:** under noise, the *reported* VQE energy is often **below** the exact ground state. That seems to violate the variational principle, but it is a selection effect: the optimiser keeps whichever noisy estimate happened to be lowest. We saw it for COBYLA at σ = 5 mHa (reported ≈ 6 mHa *below* exact on average). On hardware, always re-measure the final parameters with fresh shots.
- **Warm starts barely help here:** 169 vs 176 evaluations over the curve (4 %), with identical energies. The UCCSD amplitudes are tiny (|θ| < 0.07 rad), so the all-zero Hartree-Fock start is already close to the optimum. Warm starts should matter more for larger bases and on hardware, where every evaluation costs shots.

# 5. Hardware-aware compilation

**Claim:** the choice of encoding sets the circuit size on a real chip, and circuit size sets the noise. We compare encodings and transpiler optimisation levels, then pick the physical qubits with the lowest total reported error (gate + readout errors from the device's calibration data), scanning 20 transpiler seeds. A deliberately bad layout shows the choice matters.

Backend: `FakeFez`, a snapshot of the calibration data of IBM's 156-qubit Heron r2 device `ibm_fez`. If Section 7 runs on another device, swap in its fake copy (e.g. `FakeMarrakesh`, `FakeTorino`) so the noise model matches.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeFez
backend = FakeFez()

rows = []
for kind in ["jw", "tapered"]:
    pp = prepare_point(0.774, mapping=kind)
    for lvl in [0, 1, 2, 3]:
        r = transpile_report(pp.circuit, backend, optimization_level=lvl, seed=0)
        rows.append({"encoding": kind, "opt_level": lvl, "depth": r["depth"],
                     "two_qubit_gates": r["two_qubit_gates"], "error_score": r["error_score"]})
compile_study = pd.DataFrame(rows)
compile_study.to_csv(RESULTS / "compile_study.csv", index=False); compile_study

In [ ]:
pp_eq = prepare_point(0.774, mapping="tapered")
best = best_transpile(pp_eq.circuit, backend, n_seeds=20)
print("chosen physical qubits:", best["physical_qubits"], "| depth", best["depth"],
      "| 2q gates", best["two_qubit_gates"], "| error score", round(best["error_score"], 4))

# Deliberately BAD layout: the connected qubit pair with the worst CZ + readout errors.
t = backend.target
def pair_cost(a, b):
    return t["cz"][(a, b)].error + t["measure"][(a,)].error + t["measure"][(b,)].error
pairs = [q for q, p in t["cz"].items() if p is not None and p.error is not None and p.error < 1]  # skip broken links
worst_pair = max(pairs, key=lambda q: pair_cost(*q))
worst = transpile_report(pp_eq.circuit, backend, initial_layout=list(worst_pair))

est = noisy_estimator(backend)
rows = []
for name, r in [("noise-aware (best)", best), ("worst qubits", worst)]:
    ev, _ = estimate([r["isa"]], pp_eq.qubit_op, est, precision=0)  # precision=0: exact noisy average
    rows.append({"layout": name, "physical_qubits": r["physical_qubits"], "two_qubit_gates": r["two_qubit_gates"],
                 "error_score": r["error_score"],
                 "energy_error_mHa": 1e3 * (ev[0] + pp_eq.nuclear_repulsion - pp_eq.e_exact)})
layout_study = pd.DataFrame(rows)
layout_study.to_csv(RESULTS / "layout_study.csv", index=False)
display(layout_study)
best["isa"].draw("mpl", idle_wires=False, fold=-1)

**What this shows.**
- **Encoding matters most.** After compiling for the chip, the 4-qubit Jordan-Wigner circuit needs **45 CZ gates (depth 140)**. The 2-qubit tapered circuit needs **2 CZ gates (depth 9–11, depending on the transpiler run)**: 22× fewer two-qubit gates for the same exact answer.
- **Optimisation level matters too.** Levels 0–1 keep 4 CZs and place the circuit on qubits 0–1 without looking at their quality. Levels 2–3 simplify to 2 CZs *and* choose better qubits (error score 0.070 → 0.019).
- **Qubit choice matters.** The noise-aware layout picked qubits 22–23. On the worst connected pair (113–119, readout error ≈ 11 %), the same circuit has **17.6 mHa** error instead of **7.7 mHa**: 2.3× worse from where it runs alone. These energies include gate errors but not readout errors (see Section 6), so with its 11 % readout error the worst pair would be worse still.

# 6. Noisy simulation and error mitigation

We optimise parameters noiselessly (Section 3), then **evaluate** them under the backend's noise model. That separates noise error from optimisation error.

We use `precision=0`: Aer then returns the *exact average* energy under the noise model, with no shot noise, so what we see is purely the bias caused by noise. **Caveat we found:** this mode never simulates the measurement itself, so it includes gate and idle errors but **not readout errors** (removing readout errors from the noise model leaves the energy unchanged to 10⁻⁷ Ha). The Section 7 dry run, which really measures, includes them. Shot noise is added back in the Section 7 dry run.

ZNE: fold each 2-qubit gate (scale factors 1, 3, 5, i.e. 1×, 3×, 5× the two-qubit noise) and extrapolate back to zero noise with a straight line (linear) and a parabola (quadratic). Error bars on ZNE values are propagated with `zne_std`.

In [ ]:
from hardware_tools import zne_std
est = noisy_estimator(backend)
SCALES = [1, 3, 5]
SIM_PRECISION = 0  # 0 = exact noisy average (no shot noise)
rows = []
for d in HW_DISTANCES:
    pp = prepare_point(d, mapping="tapered")
    isa = best_transpile(pp.circuit, backend, n_seeds=10)["isa"]
    ev, sd = estimate([fold_two_qubit_gates(isa, s) for s in SCALES], pp.qubit_op, est, precision=SIM_PRECISION)
    E = ev + pp.nuclear_repulsion
    rows.append({"distance_A": d, "exact": pp.e_exact, "vqe_noiseless": pp.e_vqe,
                 "noisy": E[0], "noisy_std": sd[0],
                 "zne_linear": zne_extrapolate(SCALES, E, 1), "zne_linear_std": zne_std(SCALES, sd, 1),
                 "zne_quad": zne_extrapolate(SCALES, E, 2), "zne_quad_std": zne_std(SCALES, sd, 2),
                 **{f"scale_{s}": e for s, e in zip(SCALES, E)}})
noisy = pd.DataFrame(rows)
for c in ["noisy", "zne_linear", "zne_quad"]:
    noisy[f"{c}_err"] = (noisy[c] - noisy.exact).abs()
noisy.to_csv(RESULTS / "pes_noisy_sim.csv", index=False)
noisy[["distance_A", "exact", "noisy", "zne_linear", "zne_quad", "noisy_err", "zne_linear_err", "zne_quad_err"]]

In [ ]:
fig, (ax, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))
if "pes" in globals():  # continuous exact curve from Section 3
    ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(noisy.distance_A, noisy.noisy, "o", c="#d62728", label="Noisy sim, raw")
ax.plot(noisy.distance_A, noisy.zne_linear, "s", c="#1f77b4", label="Noisy sim + ZNE (linear)")
ax.plot(noisy.distance_A, noisy.zne_quad, "^", c="#9467bd", mfc="none", label="Noisy sim + ZNE (quadratic)")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)"); ax.set_title("Noisy simulation (FakeFez)"); ax.legend()

for c, m, col in [("noisy", "o", "#d62728"), ("zne_linear", "s", "#1f77b4"), ("zne_quad", "^", "#9467bd")]:
    ax2.semilogy(noisy.distance_A, noisy[f"{c}_err"] * 1e3, m + "-", c=col, label=c)
ax2.axhline(CHEMICAL_ACCURACY * 1e3, c="k", ls=":", label="chemical accuracy")
ax2.set_xlabel("He–H distance (Å)"); ax2.set_ylabel("|E − exact| (mHa)"); ax2.set_title("Error vs exact"); ax2.legend()

row = noisy.loc[noisy.distance_A.sub(0.774).abs().idxmin()]
E = [row[f"scale_{s}"] for s in SCALES]; x = np.linspace(0, max(SCALES), 50)
ax3.plot(SCALES, E, "o", c="#d62728", ms=8, label="measured (scales 1, 3, 5)")
for order, col in [(1, "#1f77b4"), (2, "#9467bd")]:
    ax3.plot(x, np.polyval(np.polyfit(SCALES, E, order), x), "--", c=col, label=f"{'linear' if order == 1 else 'quadratic'} fit")
ax3.axhline(row.exact, c="#2ca02c", label="exact")
ax3.set_xlabel("noise scale factor"); ax3.set_ylabel("Total energy (Ha)"); ax3.set_title(f"ZNE at {row.distance_A} Å"); ax3.legend()
fig.tight_layout(); fig.savefig(FIGURES / "noisy_sim_zne.png", dpi=150)

from hardware_tools import two_qubit_count
# Extra: same pipeline for the 4-qubit Jordan-Wigner circuit at equilibrium (deeper circuit -> more noise?)
rows = []
for kind in ["tapered", "jw"]:
    p = prepare_point(0.774, mapping=kind)
    isa = best_transpile(p.circuit, backend, n_seeds=10)["isa"]
    ev, _ = estimate([fold_two_qubit_gates(isa, s) for s in SCALES], p.qubit_op, est, precision=0)
    Ek = ev + p.nuclear_repulsion
    rows.append({"encoding": kind, "two_qubit_gates": two_qubit_count(isa),
                 "raw_err_mHa": 1e3 * abs(Ek[0] - p.e_exact),
                 "zne_linear_err_mHa": 1e3 * abs(zne_extrapolate(SCALES, Ek, 1) - p.e_exact)})
encoding_noise = pd.DataFrame(rows); encoding_noise.to_csv(RESULTS / "encoding_noise.csv", index=False)
encoding_noise

**What this shows.**
- **Gate noise alone gives 5–10 mHa of error** (3–6× chemical accuracy) even for a circuit with only 2 CZ gates. It is largest at short distances.
- **ZNE removes about 70–75 % of it**, leaving **1.3–2.6 mHa**. That is within chemical accuracy (1.6 mHa) at 1.5 Å and 2.5 Å, and just outside it at 0.6–1.0 Å.
- **Linear and quadratic agree to within 0.1 mHa.** The energy rises almost exactly in a straight line with the noise scale (right panel), so the simpler linear fit is enough. On hardware the linear fit is also the better choice because it inflates shot noise less (error bar ×1.21 vs ×2.28).
- **What is left after ZNE is not readout error**, because this mode contains none. It comes from single-qubit gate and idle errors, which folding does not amplify, plus the slight curvature a straight line cannot follow. Readout error shows up in the Section 7 dry run.
- **Deeper circuits are hopeless at this noise level.** At equilibrium, the 45-CZ Jordan-Wigner circuit is off by **121 mHa raw and 43 mHa after ZNE**, against 7.7 and 2.1 mHa for the tapered one. This is why the qubit reduction in Section 1 is essential.

# 7. Real-hardware pipeline (validated on a fake device)

This section contains everything needed to run the experiment on an IBM quantum computer, and checks it end-to-end on the fake device. **We did not run it on a real QPU within the hackathon time**, so all hardware-style numbers below come from this dry run.

**How the job is built.**
- **One job, 9 circuits:** 3 distances × 3 ZNE noise scales, so the queue is waited in only once. Each circuit is compiled for the chosen device with the noise-aware layout from Section 5.
- **Shots:** 32768 per circuit. One energy then wobbles by ≈ 4.5 mHa. With 4096 shots it would be ≈ 13 mHa, 8× chemical accuracy.
- **Error suppression and mitigation on the real device:** dynamical decoupling (XpXm) on idle qubits, Pauli twirling on gates and measurements, TREX readout mitigation (`resilience_level=1`), plus our own folding-based ZNE on top. The local dry run ignores the first three: it is a *pessimistic* preview.
- **Safe to run:** the job ID is written to `results/hardware_job.json` as soon as the job is submitted, the cell refuses to submit twice, and a second branch collects the results when the job is done.

In [ ]:
import json
from hardware_tools import build_hardware_pubs, hardware_table
HW_REAL_DISTANCES = [0.774, 1.5, 2.5]  # 3 distances x 3 noise scales = 9 circuits in ONE job
HW_SHOTS = 32768                       # ~4.5 mHa per energy (4096 shots would be ~13 mHa)

# Dry run: the identical job, on the fake device, locally.
# (Local mode ignores dynamical decoupling, twirling and TREX, so this is a pessimistic preview.)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pubs, meta = build_hardware_pubs(HW_REAL_DISTANCES, SCALES, backend)
    dry_est = runtime_estimator(backend, shots=HW_SHOTS)
    dry_est.options.simulator.seed_simulator = 1234
    dry = hardware_table(dry_est.run(pubs).result(), meta, SCALES, backend.name, "local dry run")
dry.to_csv(RESULTS / "hardware_dry_run.csv", index=False)
for c in ["hw_raw", "hw_zne_linear"]:
    dry[f"{c}_err_mHa"] = 1e3 * (dry[c] - dry.exact)
dry[["distance_A", "exact", "hw_raw", "hw_raw_err_mHa", "hw_raw_std", "hw_zne_linear", "hw_zne_linear_err_mHa", "hw_zne_linear_std"]]

In [ ]:
# Real device: submit once, then collect. Not run for this submission (RUN_HARDWARE = False in Section 0).
JOB_FILE, HW_FILE = RESULTS / "hardware_job.json", RESULTS / "hardware_pes.csv"
HW_BACKEND_NAME = "ibm_fez"  # the device FakeFez copies; needs a saved IBM Quantum account
if RUN_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    if not JOB_FILE.exists():  # submit (never twice)
        hw_backend = service.backend(HW_BACKEND_NAME)
        pubs, meta = build_hardware_pubs(HW_REAL_DISTANCES, SCALES, hw_backend)
        job = runtime_estimator(hw_backend, shots=HW_SHOTS).run(pubs)
        JOB_FILE.write_text(json.dumps({"job_id": job.job_id(), "backend": hw_backend.name, "shots": HW_SHOTS,
                                        "scales": SCALES, "submitted": time.strftime("%Y-%m-%d %H:%M"),
                                        "meta": meta}, indent=2))
        print("Submitted", job.job_id(), "on", hw_backend.name)
    elif not HW_FILE.exists():  # collect when done (safe to re-run)
        info = json.loads(JOB_FILE.read_text())
        job = service.job(info["job_id"]); status = str(job.status())
        print(info["job_id"], "->", status)
        if "DONE" in status:
            hardware_table(job.result(), info["meta"], info["scales"], info["backend"], info["job_id"]).to_csv(HW_FILE, index=False)
else:
    print("RUN_HARDWARE = False: no job sent to IBM hardware.")

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax.plot(pes.distance_A, pes.exact, "-", c="#2ca02c", label="Exact (FCI)")
ax.plot(noisy.distance_A, noisy.zne_linear, "s", c="#1f77b4", mfc="none", ms=8, label="Noisy sim + ZNE (exact average)")
ax.errorbar(dry.distance_A, dry.hw_raw, yerr=dry.hw_raw_std, fmt="o", c="#d62728", capsize=3, label=f"Dry run raw ({HW_SHOTS} shots)")
ax.errorbar(dry.distance_A, dry.hw_zne_linear, yerr=dry.hw_zne_linear_std, fmt="s", c="#1f77b4", capsize=3, label="Dry run + ZNE")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)"); ax.set_title("Hardware pipeline, dry run on FakeFez"); ax.legend(fontsize=8)
for c, std, m, col, dx in [("hw_raw", "hw_raw_std", "o", "#d62728", -0.03), ("hw_zne_linear", "hw_zne_linear_std", "s", "#1f77b4", 0.03)]:
    ax2.errorbar(dry.distance_A + dx, 1e3 * (dry[c] - dry.exact), yerr=1e3 * dry[std], fmt=m, c=col, capsize=3,
                 label="raw" if c == "hw_raw" else "ZNE (linear)")
ax2.axhspan(-CHEMICAL_ACCURACY * 1e3, CHEMICAL_ACCURACY * 1e3, color="k", alpha=0.1, label="chemical accuracy")
ax2.axhline(0, c="k", lw=0.8)
ax2.set_xlabel("He–H distance (Å)"); ax2.set_ylabel("E − exact (mHa)"); ax2.set_title("Error with shot-noise error bars"); ax2.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "hardware_dry_run.png", dpi=150)

**What the dry run shows.**
- **The pipeline works end-to-end:** 9 circuits, each compiled for the device with the noise-aware layout, sent as one job, and turned into a table with error bars.
- **Raw errors are 7.9–11.4 mHa, about 3 mHa more than Section 6 at every distance.** The difference is readout error, which only a real measurement includes.
- **ZNE brings them to 5.2–6.5 mHa, not to the 1.4–2.1 mHa of Section 6.** Folding cannot amplify readout error, so ZNE cannot remove it. Removing it is exactly the job of TREX on the real device (not available in local mode), so on hardware we expect TREX + ZNE to land closer to the Section 6 numbers.
- **Shot noise sets the error bars:** 1.9–4.5 mHa raw and 2.5–5.6 mHa after ZNE. At equilibrium, bringing the ZNE error bar under chemical accuracy would need ≈ 12× more shots (≈ 400 000 per circuit). At this budget, chemical accuracy can be shown on the simulator but not confirmed from a hardware-style run.


# 8. Performance and limitations

### Error budget at equilibrium (0.774 Å)
Computed by the cell below from `results/*.csv` (saved as `results/error_budget.csv`).

| Source | Noisy sim, exact average (Ha) | Dry run, 32768 shots (Ha) | Evidence |
|---|---|---|---|
| Basis set (6-31G − STO-3G exact energy) | −8.1 × 10⁻² | – | Section 4 |
| Ansatz / optimiser (VQE − exact, noiseless) | 5 × 10⁻⁸ | – | Section 3 |
| Shot noise (std of one energy) | 0 | 4.5 × 10⁻³ | Section 7 |
| Noise (raw − noiseless VQE) | +7.7 × 10⁻³ (gates only) | +1.14 × 10⁻² (gates + readout) | Sections 6 / 7 |
| Residual after linear ZNE (− exact) | +2.1 × 10⁻³ | +6.5 × 10⁻³ ± 5.6 × 10⁻³ | Sections 6 / 7 |

**Reading the table:** the algorithm (UCCSD + VQE) is essentially perfect. The quantum errors (noise ≈ 8–11 mHa, shot noise ≈ 5 mHa) are much bigger than the algorithm error, and readout error is the largest part ZNE leaves behind. But the **basis set error (81 mHa) is about five times bigger than all of them together**: the small STO-3G basis, not the quantum computer, limits how close we are to the real molecule. Against the near-exact cc-pVQZ FCI of Section 4a the gap is even larger: STO-3G is ~126 mHa too high at equilibrium and puts the bond 0.14 Å too long.

### Combined figure
The headline figure below shows every level on one plot: exact, noiseless VQE, noisy simulation (raw and ZNE) and the shot-based dry run of the hardware pipeline with error bars.

In [ ]:
# ---- Error budget at equilibrium (0.774 A), all numbers read from results/*.csv
eq = pd.read_csv(RESULTS / "pes_noisy_sim.csv").set_index("distance_A").loc[0.774]
dr = pd.read_csv(RESULTS / "hardware_dry_run.csv").set_index("distance_A").loc[0.774]
sc = pd.read_csv(RESULTS / "basis_encoding_scaling.csv")
basis_err = sc.query("encoding == 'tapered'").set_index("basis").exact.diff().iloc[-1]
budget = pd.DataFrame([
    {"source": "Basis set (6-31G − STO-3G exact)", "noisy_sim_Ha": basis_err, "dry_run_Ha": np.nan, "evidence": "Section 4"},
    {"source": "Ansatz / optimiser (VQE − exact, noiseless)", "noisy_sim_Ha": eq.vqe_noiseless - eq.exact, "dry_run_Ha": np.nan, "evidence": "Section 3"},
    {"source": f"Shot noise (std of one energy, {HW_SHOTS} shots)", "noisy_sim_Ha": 0.0, "dry_run_Ha": dr.hw_raw_std, "evidence": "Section 7"},
    {"source": "Gate + readout noise (raw − noiseless VQE)", "noisy_sim_Ha": eq.noisy - eq.vqe_noiseless,
     "dry_run_Ha": dr.hw_raw - dr.vqe_noiseless, "evidence": "Sections 6 / 7"},
    {"source": "Residual after ZNE (linear ZNE − exact)", "noisy_sim_Ha": eq.zne_linear - eq.exact,
     "dry_run_Ha": dr.hw_zne_linear - dr.exact, "evidence": "Sections 6 / 7"},
    {"source": "  ± error bar of that ZNE value", "noisy_sim_Ha": 0.0, "dry_run_Ha": dr.hw_zne_linear_std, "evidence": "Section 7"},
])
budget.to_csv(RESULTS / "error_budget.csv", index=False)
display(budget.style.format({"noisy_sim_Ha": "{:+.2e}", "dry_run_Ha": "{:+.2e}"}, na_rep="–"))

# ---- Headline figure: every level of the study on one plot
pes_ = pd.read_csv(RESULTS / "pes_noiseless_sto3g.csv")
ns = pd.read_csv(RESULTS / "pes_noisy_sim.csv")
dryf = pd.read_csv(RESULTS / "hardware_dry_run.csv")
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(pes_.distance_A, pes_.hf, "--", c="#9e9e9e", label="Hartree-Fock")
ax.plot(pes_.distance_A, pes_.exact, "-", c="#2ca02c", lw=2, label="Exact (FCI)")
ax.plot(pes_.distance_A, pes_.vqe, "x", c="k", ms=8, label="VQE noiseless")
ax.plot(ns.distance_A, ns.noisy, "o", c="#d62728", mfc="none", ms=8, label="Noisy sim, raw (exact average)")
ax.plot(ns.distance_A, ns.zne_linear, "s", c="#1f77b4", mfc="none", ms=8, label="Noisy sim + ZNE")
ax.errorbar(dryf.distance_A + 0.03, dryf.hw_zne_linear, yerr=dryf.hw_zne_linear_std, fmt="s", c="#1f77b4", capsize=3,
            label=f"Hardware pipeline dry run + ZNE ({HW_SHOTS} shots)")
ax.set_xlabel("He–H distance (Å)"); ax.set_ylabel("Total energy (Ha)")
ax.set_title("HeH$^+$ potential energy surface, STO-3G: from exact to noisy"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "headline_pes.png", dpi=150)

### Limitations and scaling
- **Fixed parameters.** We optimised on a noiseless simulator and only *evaluated* on noisy devices. A full VQE loop on hardware would need hundreds of jobs, and the optimiser would chase shot noise. Section 4c measured this on a simulator: with 1–5 mHa of noise per evaluation, SLSQP fails (≈ 1 Ha off), COBYLA stops a few mHa short, and only SPSA stays within chemical accuracy, at 600 evaluations (× 3 circuits each with ZNE). Noisy optimisers also *report* energies below the exact one (winner's curse), so final parameters must always be re-measured.
- **No real-device run.** The hardware pipeline is complete and tested on the fake device, but we did not run it on a real QPU within the time available. The dry run leaves out dynamical decoupling, twirling and TREX, so real-device errors could be somewhat smaller than our dry-run numbers.
- **Shot noise is the hardware bottleneck.** Even at 32768 shots one energy wobbles by several mHa, more than chemical accuracy, and ZNE makes the error bar 1.2× (linear) to 2.3× (quadratic) larger.
- **ZNE assumptions.** It assumes the energy changes smoothly and predictably as we fold gates. Folding only amplifies two-qubit gate noise, not readout or idle noise, so those leave a residual. It also costs 3× the circuits.
- **Tapering makes this system tiny.** 2 qubits and 2 CZ gates for HeH⁺/STO-3G. Bigger bases or molecules grow fast (Section 4: 6-31G already needs 6 qubits and 15 parameters after tapering; the cc-pVQZ basis that reproduces experiment would need 118), and the two-qubit gate count of UCCSD grows much faster than the qubit count. For 2 electrons classical FCI solves cc-pVQZ in seconds, so HeH⁺ is a validation benchmark, not a quantum-advantage problem.
- **Ansatz choice on hardware.** We ran UCCSD (2 CZ after compilation). Section 4b shows a 1-CNOT `real_amplitudes` circuit is also exact here, which would halve the two-qubit noise; we did not have time to test it under noise.
- **Calibration drifts.** The noise-aware layout uses calibration data that changes during the day; the qubits we picked may not stay the best.

# 9. Conclusions

**What we showed**
- **Symmetry first:** parity mapping + ℤ₂ tapering turns 4 qubits / 27 Pauli terms into 2 qubits / 9 terms with *zero* loss of accuracy, and the compiled circuit from 45 to 2 two-qubit gates. On the noisy simulator that is the difference between 121 mHa and 7.7 mHa error.
- **Validated references:** Qiskit's exact energies agree with PySCF FCI to 3×10⁻¹⁵ Ha at all 14 distances; the tutorial's unfiltered solver is 0.164 Ha off because it finds a 3-electron state.
- **VQE is exact when noiseless:** worst error 3×10⁻⁷ Ha over the curve (5000× below chemical accuracy), in STO-3G and in 6-31G. STO-3G gives r_eq = 0.914 Å and D_e = 1.49 eV; HeH⁺ dissociates to He + H⁺ because helium's ionisation energy is much higher than hydrogen's.
- **The basis set is the dominant error:** r_eq moves from 0.914 Å (STO-3G) to 0.769 Å (6-31G) to 0.775 Å (cc-pVQZ, matching the literature 0.774 Å); D_e only reaches the true 2.04 eV once polarisation functions are included.
- **Noise and mitigation:** on FakeFez a noise-aware layout halves the error compared with the worst qubits (7.7 vs 17.6 mHa), and linear ZNE removes ~70–75 % of the gate-noise error (1.4–2.6 mHa, chemical accuracy at 1.5 and 2.5 Å).
- **Hardware pipeline:** a complete one-job IBM Runtime workflow (noise-aware layout, dynamical decoupling, twirling, TREX, ZNE), validated in a 32768-shot dry run on FakeFez but not run on a real QPU. Including readout error, ZNE lands 5–6.5 mHa from exact with ±2.5–5.6 mHa error bars: readout error, which folding cannot touch, is the main residual, and shot noise prevents confirming chemical accuracy at this budget.

**With more time**
- Submit the prepared job on `ibm_fez` (one command: `RUN_HARDWARE = True`) and compare it with the dry run.
- Run the 1-CNOT `real_amplitudes` ansatz and a full SPSA optimisation loop on the device, re-measuring the final parameters with fresh shots.
- Run VQE on hardware in 6-31G (6 qubits), with measurement grouping and readout-aware ZNE (fold measurements too), to see where the noise overtakes the basis-set gain.
- Repeat the hardware job at different times of day to measure how calibration drift moves the best layout and the residual error.

# Appendix
- **Hardware:** not run on a real device. The pipeline was validated with a 32768-shot local dry run on `FakeFez` (`results/hardware_dry_run.csv`).
- **Simulations run:** 2026-11-18; full notebook ≈ 3 min on a laptop (Apple Silicon, no GPU).
- **Environment:** see the cell below.
- **Credits:** builds on B. Tirado, *Ground-State Energy of a Molecule with VQE*, BasQ Qiskit Fall Fest 2026 tutorial (CC BY 4.0 / Apache 2.0).

In [ ]:
from importlib.metadata import version
import sys
print("python", sys.version.split()[0])
for p in ["qiskit", "qiskit-nature", "qiskit-algorithms", "qiskit-aer", "qiskit-ibm-runtime", "pyscf"]:
    print(f"{p:20s}{version(p)}")